# 📰 Automatic News Summarization Using LLMs & Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srikar-111/ALT-Team-3-News-Summarization-LLM/blob/main/notebooks/News_Summarization_Colab.ipynb)

This notebook runs **T5**, **BART**, **Pegasus**, and the **Groq Cloud LLM** entirely on Google's free cloud servers with free **Nvidia T4 GPU** acceleration.

**Hardware used:** Google Cloud GPU (0% local CPU, 0 MB local disk space used).

## 1. Setup & Environment
Run this cell to clone the repository and install all required libraries.

In [ ]:
!git clone https://github.com/Srikar-111/ALT-Team-3-News-Summarization-LLM.git
%cd ALT-Team-3-News-Summarization-LLM
!pip install -q -r backend/requirements.txt
!pip install -q pandas tabulate matplotlib

## 2. Verify GPU Acceleration

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("TIP: Go to Runtime -> Change runtime type -> Select T4 GPU for 10x faster execution!")

## 3. Compare All Models on a Sample News Article
Test T5, BART, and Pegasus side by side on Google's GPU.

## 🌟 Connect this Colab Cloud GPU to your Local Website
Run this cell to start the FastAPI backend on this Colab GPU and generate a public URL.
Copy the generated `trycloudflare.com` URL and paste it into the **Connect Colab GPU** button on your website (`http://localhost:3000`).

**Benefits:**
- Runs **T5, BART, Pegasus, and LLM** on free NVIDIA T4 GPU
- 0 GB downloaded to your local laptop
- Ultra-fast inference (1-3 seconds instead of 30+ seconds on CPU)

In [ ]:
import os, sys, time, subprocess, re
from IPython.display import display, HTML

# Set environment variables for the backend
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "your_groq_api_key_here")
os.environ["GROQ_MODEL"] = "qwen/qwen3.8-27b"
os.environ["LOCAL_FILES_ONLY"] = "false"

# 1. Download Cloudflare tunnel binary if not present
if not os.path.exists("cloudflared"):
    print("📦 Downloading Cloudflare tunnel binary...")
    !wget -q -nc -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

# 2. Start FastAPI server in the background
print("🚀 Starting FastAPI backend on Colab GPU...")
backend_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# 3. Start Cloudflare tunnel
print("🌐 Creating secure public tunnel...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.3)

if public_url:
    display(HTML(f"""
    <div style="background-color: #0f172a; padding: 22px; border-radius: 12px; border: 2px solid #38bdf8; font-family: sans-serif; margin: 15px 0;">
        <h2 style="color: #38bdf8; margin: 0 0 10px 0;">🎉 Colab Cloud GPU is LIVE!</h2>
        <p style="color: #cbd5e1; font-size: 15px; margin: 0 0 12px 0;">
            Copy this public URL and paste it into your local website at <b>http://localhost:3000</b> (click <b>Connect Colab GPU</b>):
        </p>
        <div style="background: #1e293b; padding: 14px 18px; border-radius: 8px; font-size: 18px; color: #4ade80; font-family: monospace; font-weight: bold; user-select: all;">
            {public_url}
        </div>
        <p style="color: #94a3b8; font-size: 13px; margin: 12px 0 0 0;">
            ⚠️ <b>Note:</b> Keep this Colab cell running while using the website. Stop it with the Stop button when finished.
        </p>
    </div>
    """))
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n🛑 Stopping server and tunnel...")
        backend_proc.terminate()
        tunnel_proc.terminate()
else:
    print("❌ Tunnel URL not found yet. Check logs.")


In [ ]:
from backend.services.summarizers import get_summarizer

sample_article = """
The European Central Bank cut interest rates by a quarter point on Thursday, a widely expected move that 
offers some relief to consumers and businesses following a prolonged period of aggressive monetary tightening.
The ECB's Governing Council lowered its key deposit rate to 3.75% from a record high of 4.0%, marking its 
first rate cut in nearly five years. The decision comes as inflation across the 20-nation euro zone has cooled 
significantly, dropping from a peak of over 10% in late 2022 to 2.6% in May.
Based on an updated assessment of the inflation outlook, the dynamics of underlying inflation and the strength 
of monetary policy transmission, it is now appropriate to moderate the degree of monetary policy restriction, 
the ECB said in an official statement.
"""

models_to_test = ["t5", "bart", "pegasus"]

for model_name in models_to_test:
    print(f"\n{'='*20} {model_name.upper()} {'='*20}")
    summarizer = get_summarizer(model_name)
    res = summarizer.summarize(sample_article, max_length=60, min_length=20, num_beams=4)
    print(f"Summary: {res['summary']}")
    print(f"Generation Time: {res['generation_time_seconds']}s | Word Count: {res['word_count']}")

## 4. Run Quantitative Dataset Experiments (ROUGE-1, ROUGE-2, ROUGE-L)
Evaluate T5, BART, and Pegasus on the CNN/DailyMail benchmark dataset.

In [ ]:
!python scripts/run_experiment.py --dataset cnn_dailymail --num_samples 5 --models t5 bart pegasus

## 5. View Evaluation Results
Inspect the generated comparison tables and ROUGE metrics.

In [ ]:
import glob, json, pandas as pd

csv_files = glob.glob("outputs/experiments/*.csv")
if csv_files:
    latest_csv = sorted(csv_files)[-1]
    print(f"Loading results from: {latest_csv}\n")
    df = pd.read_csv(latest_csv)
    display(df[['model', 'rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'avg_generation_time_seconds']])
else:
    print("Run step 4 above to generate experiment results!")